# Megatron-LM 3D Parallelism Tutorial

## Overview

Megatron-LM combines tensor, pipeline, and data parallelism for training models with hundreds of billions of parameters.

### Learning Objectives
- Understand 3D parallelism architecture
- Configure parallel dimensions
- Analyze communication patterns

### References
- Shoeybi et al., "Megatron-LM: Training Multi-Billion Parameter Language Models", arXiv 2019
- Narayanan et al., "Efficient Large-Scale Language Model Training on GPU Clusters", SC 2021

## 1. 3D Parallelism Architecture

```
Total GPUs = TP × PP × DP

Example: 64 GPUs with TP=4, PP=4, DP=4

┌─────────────────────────────────────────────────────────┐
│                    Data Parallel (DP=4)                 │
│  ┌─────────────┐ ┌─────────────┐ ┌─────────────┐ ┌────┐│
│  │  Replica 0  │ │  Replica 1  │ │  Replica 2  │ │ R3 ││
│  │             │ │             │ │             │ │    ││
│  │ ┌─────────┐ │ │ ┌─────────┐ │ │             │ │    ││
│  │ │Pipeline │ │ │ │Pipeline │ │ │    ...      │ │... ││
│  │ │ PP=4    │ │ │ │ PP=4    │ │ │             │ │    ││
│  │ │┌──┬──┬─┐│ │ │ │         │ │ │             │ │    ││
│  │ ││S0│S1│..││ │ │ │         │ │ │             │ │    ││
│  │ │└──┴──┴─┘│ │ │ │         │ │ │             │ │    ││
│  │ │ TP=4    │ │ │ │         │ │ │             │ │    ││
│  │ └─────────┘ │ │ └─────────┘ │ │             │ │    ││
│  └─────────────┘ └─────────────┘ └─────────────┘ └────┘│
└─────────────────────────────────────────────────────────┘
```

In [ ]:
from dataclasses import dataclass

@dataclass
class MegatronConfig:
    """Megatron 3D parallelism configuration."""
    tensor_model_parallel_size: int = 1  # TP
    pipeline_model_parallel_size: int = 1  # PP
    data_parallel_size: int = 1  # DP
    
    @property
    def world_size(self) -> int:
        return self.tensor_model_parallel_size * self.pipeline_model_parallel_size * self.data_parallel_size

def recommend_parallelism(num_gpus: int, model_size_b: float):
    """Recommend parallelism configuration."""
    if model_size_b < 10:
        tp, pp = 1, 1
    elif model_size_b < 50:
        tp, pp = min(4, num_gpus), 1
    elif model_size_b < 200:
        tp, pp = min(8, num_gpus), min(4, num_gpus // 8) or 1
    else:
        tp, pp = 8, min(8, num_gpus // 8)
    
    dp = num_gpus // (tp * pp)
    print(f"Model: {model_size_b}B params, {num_gpus} GPUs")
    print(f"Recommended: TP={tp}, PP={pp}, DP={dp}")
    return tp, pp, dp

recommend_parallelism(64, 175)  # GPT-3 scale

## 2. Communication Analysis

| Parallelism | Communication | Frequency |
|-------------|---------------|----------|
| Tensor (TP) | AllReduce | Per layer |
| Pipeline (PP) | P2P | Per micro-batch |
| Data (DP) | AllReduce | Per step |

## 3. Summary

### Key Takeaways

1. **TP**: Split layers, high bandwidth needed (intra-node)
2. **PP**: Split model depth, moderate bandwidth
3. **DP**: Replicate model, gradient sync
4. **Optimal**: TP within node, PP across nodes, DP for scaling